**Silver Tranformation**

In [4]:
today_date = '2026-08-09'

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 6, Finished, Available, Finished, False)

In [5]:
Fabric_Bronze_Path = 'abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_bronze'


from pyspark.sql.functions import col
df = spark.read.format('delta').load(Fabric_Bronze_Path).filter(col('processing_date')==str(today_date))

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 7, Finished, Available, Finished, False)

In [6]:
display(df)

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69b4b9fd-7bc7-4ab0-a731-add1866b165a)

In [7]:
## Data Cleanning
print('before removing duplicate', df.count())
df_remove_duplicate = df.dropDuplicates()
print('after removing duplicate', df.count())

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 9, Finished, Available, Finished, False)

before removing duplicate 1675
after removing duplicate 1675


In [8]:
## handling missing values
df_dropped = df_remove_duplicate.dropna(subset= ['Order_ID','Customer_ID'])

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 10, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import col, datediff

df_days = df_dropped.withColumn(
    "Delivery_Days",
    datediff(col("Ship_Date"), col("Order_Date"))
)

display(df_days)

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad0a4207-779d-44c8-940b-0d5fb922022d)

In [10]:
from pyspark.sql.functions import col, try_divide, round
df_profit_margin = df_days.withColumn(
    "Profit_Margin",
    round(try_divide(col("Profit"), col("Sales")) * 100, 2)
)

display(df_profit_margin)

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 400192a3-bb30-4511-ac9a-298d26300dfa)

In [11]:
df_profit_margin.createOrReplaceTempView("t_silver_new_data")

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 13, Finished, Available, Finished, False)

In [12]:
%%sql
select * from t_silver_new_data

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 29 fields>

In [15]:
Fabric_tbl_silver = "abfss://Fabric_Dev@onelake.dfs.fabric.microsoft.com/Fabric_LH_Sales.Lakehouse/Tables/dbo/tblsales_silver"

try:
    spark.read.format("delta") \
        .load(Fabric_tbl_silver) \
        .createOrReplaceTempView("t_tblsales_silver")

except:

    v_create_table = """
    CREATE TABLE IF NOT EXISTS tblsales_silver (
        Row_ID STRING,
        Order_ID STRING,
        Order_Date DATE,
        Ship_Date DATE,
        Ship_Mode STRING,
        Customer_ID STRING,
        Customer_Name STRING,
        Segment STRING,
        Postal_Code STRING,
        City STRING,
        State STRING,
        Country STRING,
        Region STRING,
        Market STRING,
        Product_ID STRING,
        Category STRING,
        Sub_Category STRING,
        Product_Name STRING,
        Sales DOUBLE,
        Quantity INT,
        Discount DOUBLE,
        Profit DOUBLE,
        Shipping_Cost DOUBLE,
        Order_Priority STRING,
        Month STRING,
        Year STRING,
        processing_date DATE,
        Delivery_Days INT,
        Profit_Margin DOUBLE
    )
    """

    spark.sql(v_create_table)

    spark.read.format("delta") \
        .load(Fabric_tbl_silver) \
        .createOrReplaceTempView("t_tblsales_silver")

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 17, Finished, Available, Finished, False)

In [16]:
display(
    spark.sql("""
        SELECT *
        FROM t_tblsales_silver
    """)
)

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 573da6a0-4283-4f47-9600-ea803e6fdee1)

In [17]:
sql_statement = f"""MERGE INTO tblsales_silver as target
        USING t_silver_new_data as source
        on target.Order_ID = source.Order_ID and target.Customer_ID = source.Customer_ID

        WHEN MATCHED THEN
            UPDATE SET
            target.Row_ID = source.Row_ID,
            target.Order_ID  = source.Order_ID,
            target.Order_Date = source.Order_Date,
            target.Ship_Date  = source.Ship_Date,
            target.Ship_Mode  =  source.Ship_Mode,
            target.Customer_ID = source.Customer_ID,
            target.Customer_Name  = source.Customer_Name,
            target.Segment  = source.Segment,
            target.Postal_Code  = source.Postal_Code ,
            target.City  = source.City,
            target.State  = source.State ,
            target.Country  = source.Country ,
            target.Region  = source.Region,
            target.Market  = source.Market ,
            target.Product_ID  = source.Product_ID,
            target.Category  = source.Category,
            target.Sub_Category  = source.Sub_Category ,
            target.Product_Name  = source.Product_Name,
            target.Sales  = source.Sales,
            target.Quantity  = source.Quantity ,
            target.Discount  = source.Discount,
            target.Profit  = source.Profit,
            target.Shipping_Cost  = source.Shipping_Cost,
            target.Order_Priority  = source.Order_Priority,
            target.Month  = source.Month,
            target.Year  = source.Year ,
            target.processing_date = source.processing_date,
            target.Delivery_Days = source.Delivery_Days,
            target.Profit_Margin = source.Profit_Margin


        WHEN NOT MATCHED THEN
            INSERT (Row_ID,
                    Order_ID,
                    Order_Date,
                    Ship_Date,
                    Ship_Mode,
                    Customer_ID,
                    Customer_Name,
                    Segment,
                    Postal_Code,
                    City,
                    State,
                    Country,
                    Region,
                    Market,
                    Product_ID,
                    Category,
                    Sub_Category,
                    Product_Name,
                    Sales,
                    Quantity,
                    Discount,
                    Profit,
                    Shipping_Cost,
                    Order_Priority,
                    Month,
                    Year,
                    processing_date,
                    Delivery_Days,
                    Profit_Margin)
            VALUES (source.Row_ID,
                    source.Order_ID,
                    source.Order_Date,
                    source.Ship_Date,
                    source.Ship_Mode,
                    source.Customer_ID,
                    source.Customer_Name,
                    source.Segment,
                    source.Postal_Code,
                    source.City,
                    source.State,
                    source.Country,
                    source.Region,
                    source.Market,
                    source.Product_ID,
                    source.Category,
                    source.Sub_Category,
                    source.Product_Name,
                    source.Sales,
                    source.Quantity,
                    source.Discount,
                    source.Profit,
                    source.Shipping_Cost,
                    source.Order_Priority,
                    source.Month,
                    source.Year,
                    source.processing_date,
                    source.Delivery_Days,
                    source.Profit_Margin
                    )"""
spark.sql(sql_statement).show()

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 19, Finished, Available, Finished, False)

+-----------------+----------------+----------------+-----------------+
|num_affected_rows|num_updated_rows|num_deleted_rows|num_inserted_rows|
+-----------------+----------------+----------------+-----------------+
|             1675|               0|               0|             1675|
+-----------------+----------------+----------------+-----------------+



In [18]:
%%sql
select * from tblsales_silver

StatementMeta(, 64f52a4f-fab1-47de-b561-74d72902f6a4, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 29 fields>